# Strategy 2. Template-based strategy

Using BioMix & pre-defined templates


In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [2]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

def ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

In [3]:
# Json data extraction

import re

def extract_json(message):
    """Extract JSON from LLM response - handles both markdown and direct JSON"""
    
    if not message:
        raise ValueError("Empty message")
    
    # Extract from markdown code blocks (```json ... ```)
    markdown_pattern = r"```(?:json)?\s*(.*?)```"
    matches = re.findall(markdown_pattern, message, re.DOTALL)
    
    if matches:
        try:
            return [json.loads(match.strip()) for match in matches]
        except Exception as e:
            print(f"Warning: Found markdown block but failed to parse: {e}")
    
    # Parse entire message as JSON (for direct JSON output like GPT-5 and later)
    try:
        parsed = json.loads(message.strip())
        return [parsed]  # Return as list for consistency
    except json.JSONDecodeError:
        pass  # Not direct JSON, continue to next attempt
    
    # Find JSON objects anywhere in the text
    # This regex finds balanced curly braces
    json_pattern = r'\{(?:[^{}]|(?:\{(?:[^{}]|(?:\{[^{}]*\}))*\}))*\}'
    json_matches = re.findall(json_pattern, message, re.DOTALL)
    
    for match in json_matches:
        try:
            parsed = json.loads(match)
            # Validate it has the required structure
            if isinstance(parsed, dict) and 'query_name' in parsed:
                return [parsed]
        except:
            continue
    
    raise ValueError(f"Failed to parse JSON from message: {message[:200]}...")


    

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)


In [4]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

### Generating cypher query from query name and parameters in json

We'll hard-code cypher queries generation in this strategy. LLM should produce the output like this:

```json
{
    "query_name": "gene_disease_association",
    "query_params": {
        "gene_symbol" : "TARDBP",
        "disease_name" : ["amyotrophic lateral sclerosis", "als"],
        "evidence_type" : ["Literature", "AnimalModel"],
        "evidence_threshold" : 0.3
    }
}
```

Cypher queries will be generated using query_params.

In [5]:
# generating query from json schema

template_gene_disease_query = """
MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = '{gene_symbol}' 
  AND {disease_names}
  AND assoc.score IS NOT NULL
  {evidence_type}
  {score_threshold}
RETURN 
    gene.approvedSymbol as Gene,
    disease.name as Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    assoc.score as Score,
    assoc.literature as Literature
ORDER BY assoc.score DESC

"""



def generate_cypher_gene_disease_association(params):
    
    gene_symbol = params['gene_symbol']

    # disease:
    # (LOWER(disease.name) CONTAINS LOWER('amyotrophic lateral sclerosis') OR LOWER(disease.name) CONTAINS LOWER('ALS'))    
    diseases = params['disease_name']
    if len(diseases) == 0:
        raise ValueError(f"diseases list should not be empty in query: {params}")
    else:
        disease_names = "(" + " OR ".join([f'LOWER(disease.name) CONTAINS "{d.lower()}"' for d in diseases]) + ")"
    
    # evidence_type:
    #AND (assoc:`AnimalModel.GeneToDiseaseAssociation` OR assoc:`Literature.GeneToDiseaseAssociation`)
    evidence_type = ""
    if 'evidence_type' in params:
        if len(params['evidence_type']) > 0:
            evidence_type = "AND (" +  " OR ".join([f"assoc:`{t}.GeneToDiseaseAssociation`" for t in params['evidence_type']])  + ")"

    # score_threshold:
    #AND assoc.score > 0.3
    score_threshold = ""
    if 'score_threshold' in params:
        score_threshold = f"AND assoc.score > {params['score_threshold']}"
    
    return template_gene_disease_query.format(gene_symbol=gene_symbol, disease_names=disease_names, evidence_type=evidence_type, score_threshold=score_threshold)
    
        
def generate_cypher_mock_query(query_name, params):
    raise ValueError(f"Requested decoy query {query_name}")


def generate_cypher(query):

    query_name = query['query_name']
    query_params = query['query_params']

    if query_name == 'gene_disease_association':
        return generate_cypher_gene_disease_association(query_params)
    elif query_name == 'drug_target_association':
        return generate_cypher_mock_query(query_name, query_params)
    elif query_name == 'drug_disease_association':
        return generate_cypher_mock_query(query_name, query_params)
    
    else:
        raise ValueError(f"Incorrect query name {query_name}")
    
    


In [6]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        json_queries = extract_json(llm_output)
        cypher_query = [generate_cypher(json_query) for json_query in json_queries]
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

Loading from an extended biomix test-set

In [7]:
questions = pd.read_csv("../biomix/testset/biomix_true_false_selected_augmented.csv")

## Running template-based query

We'll make 3 queries: 
1) with a single option (to understand if LLM picks parameters correctly), 
2) with many options and examples (to see how presence of other queries confuses LLM)
3) with many options, and without examples

The overall approach could be easily generalized to enable support of validated json schemas

In [8]:
from langchain_core.prompts import PromptTemplate


# Query building blocks:

gene_description = """\
field_name: gene_symbol
field_type: mandatory, string
field_value: official HGCN-approved symbol of human gene, e.g. "KRAS"\
"""

disease_description = """\
field_name: disease_name
field_type: mandatory, list of strings
field_value: a non-empty list of diseases that you want to retrieve an association with. for example, ["lung cancer", "colon cancer"]. The algorithm with match substrings, so if you are not sure about correct spelling, use all possible variants you can think of. Search is case-insensitive.\
"""

evidence_type_description = """\
field_name: evidence_type
field_type: optional, list of strings
field_value: specify if you want to restrict associations by evidence type. List all evidence types you want to include. For gene-disease associations allowed values are: AffectedPathway, AnimalModel, GeneticAssociation, KnownDrug, Literature, RnaExpression, SomaticMutation. If you don't want restriction, omit this field. Note that other fields may have other allowed evidence types.\
"""

evidence_threshold_description = """\
field_name: evidence_threshold
field_type: optional, double between 0 and 1
field_value: specify the threshold for evidence score. The lower score the worse the evidence. Provide minimal value for the evidence to be printed out. If you omit the value, all associations will be printed.\
"""

# Output examples:

query_example_gene_disease_association = """\
Input: Associations between KRAS and cancer, rna expression only, and evidence score >= 0.1
Output: 

```json
{
    "query_name": "gene_disease_association",
    "query_params": {
        "gene_symbol" : "KRAS",
        "disease_name" : ["cancer", "carcinoma"],
        "evidence_type" : ["RnaExpression"],
        "evidence_threshold" : 0.1
    }
}
```\
"""

query_example_drug_target_association = """\
Input: Clinically investigated or approved HDAC1 drugs.
Output: 

```json
{
    "query_name": "drug_target_association",
    "query_params": {
        "target_name" : "HDAC1",
        "evidence_type" : ["ApprovedDrug", "ClinicalStudy"]
    }
}
```\
"""

query_example_drug_indication_association = """\
Input: Approved indications of crizotinib.
Output: 

```json
{
    "query_name": "drug_disease_association",
    "query_params": {
        "drug_name" : "crizotinib",
        "evidence_type" : ["ApprovedDrug"]
    }
}
```\
"""

# Descriptions:

description_gene_disease_association = f"""\
**gene_disease_association**

This query returns associations between genes and diseases, and types of evidence

{gene_description}

{disease_description}

{evidence_type_description}

{evidence_threshold_description}\
"""

description_drug_target_association = f"""\
**drug_target_association**

This query returns associations between drugs and their targets

field_name: drug_name
field_type: mandatory, string
field_value: name of the drug. If not specified, query will return all drugs associated with a target

field_name: target_name
field_type: mandatory, string
field_value: official HGCN-approved symbol of human target, e.g. "KRAS". If not specified, query will return all targets associated with drugs.

field_name: evidence_type
field_type: optional, list of strings
field_value: specify if you want to restrict associations by evidence type. List all evidence types you want to include. Allowed values are: ClinicalStudy, AnimalModel, InVitroModel, ApprovedDrug. If you don't want restriction, omit this field.  Note that other fields may have other allowed evidence types.

{evidence_threshold_description}\
"""

description_drug_disease_association = f"""\
**drug_disease_association**

This query returns associations between drugs and diseases they intend to treat

field_name: drug_name
field_type: mandatory, string
field_value: name of the drug

{disease_description}

field_name: evidence_type
field_type: optional, list of strings
field_value: specify if you want to restrict associations by evidence type. List all evidence types you want to include. Allowed values are: ClinicalStudy, AnimalModel, InVitroModel, ApprovedDrug. If you don't want restriction, omit this field.  Note that other fields may have other allowed evidence types.\
"""

description_single_query = f"""\
{description_gene_disease_association}\
"""

description_multiple_queries = f"""\
{description_gene_disease_association}

{description_drug_target_association}

{description_drug_disease_association}\
"""

example_single_query = f"""\
Here is an example of correct json:

{query_example_gene_disease_association}
--------------------------------------------    
"""

example_multiple_query = f"""\
Here are few examples of correct json:

{query_example_gene_disease_association}

{query_example_drug_target_association}

{query_example_drug_indication_association}
--------------------------------------------    
"""



system_prompt_generic = """
You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a database with biological data that accepts queries in a json format and returns the results.

The json should have 2 mandatory fields: query_name and query_params:

field_name: query_name
field_value: name of the query. See allowed values and descriptions below

field_name: query_params
field_value: a dictionary with parameters specific for each query (see below)

Here are parameters for each individual query:

--------------------------------------------
{query_descriptions}
--------------------------------------------
{examples}

Scientists will give you question, and you will generate a json query that will help to answer the question
"""



system_prompt_multiple_queries = system_prompt_generic.format(query_descriptions = description_multiple_queries, examples = example_multiple_query)
system_prompt_extra = f"""
{system_prompt_multiple_queries}

NOTE: Just generate json to check the correctness of the statement provided by scientist! Even if you think the statement is incorrect, you still need to generate json query to double check.
"""


template_query = """
{question}
"""

user_template = PromptTemplate.from_template(template_query)


# Option 2b - multiple queries

One correct query and 2 decoy queries + json examples

In [9]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage
import openai
from anthropic import Anthropic

# Define base models to test
models_without_effort = ["gpt-4o"]  # does not have reasoning effort
models_with_effort = ["gpt-5.2-2025-12-11", "claude-sonnet-4-5-20250929"]

# Define reasoning effort levels
reasoning_efforts = ["medium", "high"]

# Create model configurations with effort levels
# Format: "model_name|effort_level" (e.g., "gpt-4o|medium", "gpt-4o|high")
models = models_without_effort + [f"{model}|{effort}" for model in models_with_effort for effort in reasoning_efforts]

niter = 1

# Create todo list: [(model_with_effort, question), ...]
todo = [(m, r['text']) for _, r in questions.iterrows() for m in models for _ in range(niter)]

def run_llm2b(model_with_effort, question):

    try:
        
        #Check if this is a model with effort level (gpt-4o stays the same)
        if "|" in model_with_effort:
            model_name, effort = model_with_effort.split("|")
        else:
            model_name = model_with_effort # For GPT-4o
            effort = None
        
        if model_name == 'gpt-4o':
            llm = ChatModel(model = model_name)

            messages = [
                SystemMessage(content=system_prompt_multiple_queries),
                HumanMessage(content=question)
            ]
            result = llm.invoke(messages)
            return result.content
            
        # For OpenAI reasoningmodels
        elif model_name.startswith("gpt-5"):
            client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt_multiple_queries},
                    {"role": "user", "content": question}
                ],
                reasoning_effort=effort
            )
            return response.choices[0].message.content
        
        # For Claude models with extended thinking
        elif model_name.startswith("claude"):
            client = Anthropic()
            
            # Build messages - prefill thinking block for extended reasoning
            messages = [{"role": "user", "content": question}]
    
            
            response = client.messages.create(
                model=model_name,
                max_tokens=16000,  # Extended thinking may need more tokens
                system=system_prompt_multiple_queries,
                messages=messages,
                thinking={
                    "type": "enabled",
                    "budget_tokens": 12000 if effort == "high" else 6000
                }
            )      
        
            # Extract text content (thinking blocks are separate)
            text_content = ""
            for block in response.content:
                if block.type == "text":
                    text_content += block.text
            
            return text_content
        
        else:
            raise ValueError(f"Unsupported model: {model_name}")
            
    except Exception as e:
        print(f"Error with {model_with_effort}: {e}")
        return None

In [10]:
llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm2b(llm_model, question))

Prompting LLM: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [37:40<00:00,  4.52s/it]


In [11]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph:  74%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                | 368/500 [01:03<00:14,  8.91it/s]

Querying graph: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [01:13<00:00,  6.78it/s]


In [12]:
results = process_results(todo, llm_answers, cypher_results)
with open("../Thinking_models_only/results/biomix02b-evaluations_ThinkingModles.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

In [13]:
results_df = pd.DataFrame(results)
results_df["has_interaction"] = results_df["count"] > 0
results_df = results_df.merge(questions, left_on="question", right_on="text", how="left")
results_df = results_df.drop(columns=["text"])
results_df['direct'] = ~results_df['question'].str.contains("is not associated")
results_df["answer"] = results_df["direct"] == results_df["has_interaction"]

In [14]:
results_df.to_excel("../Thinking_models_only/results/biomix02b-evaluations.xlsx", index=False)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error,has_interaction,label,direct,answer
0,gpt-4o,Polycythemia Vera is not associated with Gene ...,To investigate whether there is an association...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(JAK2, polycythemia vera, GeneticAssociation....",0.179487,517.0,NaN,True,False,False,False
1,gpt-5.2-2025-12-11|medium,Polycythemia Vera is not associated with Gene ...,"```json\n{\n ""query_name"": ""gene_disease_asso...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(JAK2, polycythemia vera, GeneticAssociation....",0.141154,517.0,NaN,True,False,False,False
2,gpt-5.2-2025-12-11|high,Polycythemia Vera is not associated with Gene ...,"```json\n{\n ""query_name"": ""gene_disease_asso...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(JAK2, polycythemia vera, GeneticAssociation....",0.273212,517.0,NaN,True,False,False,False
3,claude-sonnet-4-5-20250929|medium,Polycythemia Vera is not associated with Gene ...,"Based on your statement, I'll generate a query...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(JAK2, polycythemia vera, GeneticAssociation....",0.203230,587.0,NaN,True,False,False,False
4,claude-sonnet-4-5-20250929|high,Polycythemia Vera is not associated with Gene ...,"Based on your statement, I'll generate a query...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,"[(JAK2, polycythemia vera, GeneticAssociation....",0.086053,587.0,NaN,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,gpt-4o,Smith-Lemli-Opitz Syndrome is associated with ...,To address the question of whether there is an...,[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.072830,0.0,NaN,False,False,True,False
496,gpt-5.2-2025-12-11|medium,Smith-Lemli-Opitz Syndrome is associated with ...,"```json\n{\n ""query_name"": ""gene_disease_asso...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.078277,0.0,NaN,False,False,True,False
497,gpt-5.2-2025-12-11|high,Smith-Lemli-Opitz Syndrome is associated with ...,"```json\n{\n ""query_name"": ""gene_disease_asso...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.071954,0.0,NaN,False,False,True,False
498,claude-sonnet-4-5-20250929|medium,Smith-Lemli-Opitz Syndrome is associated with ...,"Based on your statement, here's a JSON query t...",[{'query': ' MATCH (gene:HumanGene)-[:IS_PART_...,1,\nMATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc...,True,[],0.053207,0.0,NaN,False,False,True,False


In [15]:
# Calculate the fraction of correct answers for each model
accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()
accuracy_df.columns = ['model', 'accuracy']
accuracy_df

/var/folders/xh/h0s_k2yj7vl5lcl4dyjlstxr0000gp/T/ipykernel_29019/2029851061.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()


,model,accuracy
0,claude-sonnet-4-5-20250929|high,0.98
1,claude-sonnet-4-5-20250929|medium,0.97
2,gpt-4o,0.96
3,gpt-5.2-2025-12-11|high,0.94
4,gpt-5.2-2025-12-11|medium,0.97
